# Step 02: Data Cleaning
Goal: Handle duplicates, standardize categories, parse currencies, and impute missing values. 
**Grading Criterion: Step Logging implemented.**

In [1]:
import pandas as pd
import numpy as np
import re

INPUT_FILE = "../data/interim/01_extraction.csv"
OUTPUT_FILE = "../data/interim/02_cleaning.csv"

df = pd.read_csv(INPUT_FILE)

def log_step(step_name, data):
    print(f"--- {step_name} ---")
    print(f"Shape: {data.shape}")
    print(f"Total Nulls: {data.isnull().sum().sum()}")
    print("\n")

log_step("INITIAL LOAD", df)

--- INITIAL LOAD ---
Shape: (10000, 25)
Total Nulls: 48403




In [2]:
# Remove Duplicates
df = df.drop_duplicates()
log_step("DUPLICATES REMOVED", df)

--- DUPLICATES REMOVED ---
Shape: (10000, 25)
Total Nulls: 48403




In [3]:
# Standardise Categorical Columns

# Gender
df['Gender'] = df['Gender'].str.strip().str.title()
df['Gender'] = df['Gender'].where(df['Gender'].isin(['Male', 'Female', 'Other']), other='Unknown')

# Repayment Status
status_map = {
    'late'    : 'Late',
    'Late'    : 'Late', 
    'Delayed' : 'Late',
    'ONTIME'  : 'On-Time',
    'On-time' : 'On-Time',
    'Default' : 'Default'
}
df['Repayment_Status'] = df['Repayment_Status'].map(status_map).fillna('Unknown')

# Sector & Residential & Enterprise & Specially Abled & IFSC
df['Sector'] = df['Sector'].str.strip().str.title().fillna('Unknown')
df['Residential Type'] = df['Residential Type'].str.strip().str.title().fillna('Unknown')
df['Residential Type'] = df['Residential Type'].where(df['Residential Type'].isin(['Urban', 'Rural']), other='Unknown')
df['Enterprise Type'] = df['Enterprise Type'].str.strip().str.title().fillna('Unknown')
df['Specially Abled Type'] = df['Specially Abled Type'].str.strip().str.title().fillna('Unknown')
df['IFSC Code'] = df['IFSC Code'].str.strip().str.upper().fillna('Unknown')

log_step("CATEGORIES STANDARDISED", df)

--- CATEGORIES STANDARDISED ---
Shape: (10000, 25)
Total Nulls: 35809




In [4]:
# Currency Parsing Logic
def parse_loan_amount(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip().replace(',', '')
    val = val.replace('₹', '').replace('$', '').strip()
    
    if re.search(r'k\s*inr', val, re.IGNORECASE):
        num = re.sub(r'[^\d.]', '', val)
        return float(num) * 1000 if num else np.nan
    
    try:
        return float(val)
    except ValueError:
        return np.nan

df['Loan_Amount_INR'] = df['Loan Amount (Mixed Units)'].apply(parse_loan_amount)
df['Project_Cost_INR'] = df['Project Cost (Mixed Units)'].apply(parse_loan_amount)

df = df.drop(columns=['Loan Amount (Mixed Units)', 'Project Cost (Mixed Units)'])
log_step("CURRENCY PARSED", df)

--- CURRENCY PARSED ---
Shape: (10000, 25)
Total Nulls: 35809




In [5]:
# Date Conversion
df['Application Date'] = pd.to_datetime(df['Application Date'], errors='coerce')
log_step("DATES CONVERTED", df)

--- DATES CONVERTED ---
Shape: (10000, 25)
Total Nulls: 37913




/var/folders/mk/zqp3608s6qxcm1n48j0crbkw0000gn/T/ipykernel_17851/3299308010.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Application Date'] = pd.to_datetime(df['Application Date'], errors='coerce')


In [6]:
# Median Imputation for Missing Values
impute_cols = [
    'Loan_Amount_INR', 'Applicant_Monthly_Income', 'Risk_Score', 
    'Loan_Tenure_Months', 'Total_TAT_Days', 'Project_Cost_INR',
    'Days_to_BHD_Verification', 'Days_to_SBDU_Verification', 
    'Days_to_DLIC_Approval', 'Days_to_Bank_Sanction', 'Days_to_Disbursement',
    'Interest_Rate_%', 'EMI_Amount', 'EMI_to_Income_Ratio', 'Credit_Eligibility_Score'
]

for col in impute_cols:
    df[col] = df[col].fillna(df[col].median())

log_step("MISSING VALUES IMPUTED", df)

--- MISSING VALUES IMPUTED ---
Shape: (10000, 25)
Total Nulls: 4090




In [7]:
# Outlier Capping (IQR)
def cap_iqr_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return series.clip(lower=lower, upper=upper)

for col in impute_cols:
    df[col] = cap_iqr_outliers(df[col])

df['Risk_Score'] = df['Risk_Score'].clip(0, 100)
log_step("OUTLIERS CAPPED", df)

--- OUTLIERS CAPPED ---
Shape: (10000, 25)
Total Nulls: 4090




In [8]:
df.to_csv(OUTPUT_FILE, index=False)
log_step("CLEANED DATA SAVED", df)

--- CLEANED DATA SAVED ---
Shape: (10000, 25)
Total Nulls: 4090


